# EEGNet SSVEP Classification

Within-subject and cross-subject SSVEP classification using EEGNet, evaluating accuracy across three dimensions: frequency, transparency, and color.

In [ ]:
import os
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, read_raw_bids
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'Times New Roman'

BIDS_ROOT = 'dataset/derivatives/preprocessed'
TASK = 'ssvep'
DATATYPE = 'eeg'
SUFFIX = 'eeg'

TARGET_FREQS = [10.0, 12.0, 15.0]
ANALYSIS_TMIN = 0.5
ANALYSIS_TMAX = 4.5
MAX_ANALYSIS_FREQ = 100.0

OCCIPITAL_CHS = ['O1', 'Oz', 'O2', 'Pz', 'P3', 'P4']

AUX_CHANNEL_TYPES = {
    'REOG': 'eog', 'LEOG': 'eog',
    'EMG1': 'emg', 'EMG2': 'emg', 'EMG3': 'emg', 'EMG4': 'emg', 'EMG5': 'emg',
    'EMG6': 'emg', 'EMG7': 'emg', 'EMG8': 'emg', 'EMG9': 'emg',
    'SpO2': 'misc', 'PulseRate': 'misc', 'Pleth': 'misc',
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. EEGNet Model

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes,
                 F1=8, D=2, F2=16, dropout_rate=0.5):
        super(EEGNet, self).__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F2

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate),
        )

        self._compute_flatten_size(n_channels, n_samples)
        self.classifier = nn.Linear(self._flatten_size, n_classes)

    def _compute_flatten_size(self, n_channels, n_samples):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.block1(dummy)
            x = self.block2(x)
            x = self.block3(x)
            self._flatten_size = x.view(1, -1).shape[1]

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.flatten(1)
        return self.classifier(x)

## 2. Utility Functions

In [ ]:
def parse_stim_label(event_name):
    event_name = str(event_name)
    parts = event_name.split('_')
    if len(parts) != 4 or parts[0] != 'stim':
        raise ValueError(f'Not a valid stim label: {event_name}')
    return {
        'Event': event_name,
        'Frequency': float(parts[1].replace('Hz', '')),
        'Transparency': int(parts[2].replace('trans', '')),
        'Color': parts[3],
    }


def discover_subjects(bids_root):
    subjects = []
    for entry in sorted(os.listdir(bids_root)):
        if entry.startswith('sub-') and os.path.isdir(os.path.join(bids_root, entry)):
            eeg_dir = os.path.join(bids_root, entry, 'eeg')
            if os.path.isdir(eeg_dir):
                subjects.append(entry.replace('sub-', ''))
    return subjects


def load_subject_data(subject_id):
    bids_path = BIDSPath(
        subject=subject_id, task=TASK, suffix=SUFFIX,
        datatype=DATATYPE, root=BIDS_ROOT,
    )
    raw = read_raw_bids(bids_path=bids_path, verbose=False)
    raw.load_data()

    present_aux_types = {ch: ch_type for ch, ch_type in AUX_CHANNEL_TYPES.items() if ch in raw.ch_names}
    if present_aux_types:
        try:
            raw.set_channel_types(present_aux_types, on_unit_change='ignore')
        except TypeError:
            raw.set_channel_types(present_aux_types)

    all_events, all_event_dict = mne.events_from_annotations(raw, verbose=False)
    event_dict = {str(name): code for name, code in all_event_dict.items()}
    stim_event_id = {name: code for name, code in event_dict.items() if name.startswith('stim_')}

    if not stim_event_id:
        return None

    epochs = mne.Epochs(
        raw, all_events, event_id=stim_event_id,
        tmin=ANALYSIS_TMIN, tmax=ANALYSIS_TMAX,
        baseline=None, preload=True, verbose=False,
    )

    available_chs = [ch for ch in OCCIPITAL_CHS if ch in epochs.ch_names]
    if not available_chs:
        return None

    epochs.pick_channels(available_chs)

    data = epochs.get_data()
    sfreq = epochs.info['sfreq']

    id_to_name = {code: name for name, code in epochs.event_id.items()}
    metadata = []
    for event_code in epochs.events[:, 2]:
        info = parse_stim_label(id_to_name[int(event_code)])
        metadata.append(info)

    return {
        'data': data,
        'metadata': pd.DataFrame(metadata),
        'n_channels': len(available_chs),
        'n_samples': data.shape[2],
        'sfreq': sfreq,
    }


def normalize_data(X_train, X_test):
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

## 3. Training & Evaluation Functions

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs=100, lr=0.001, patience=15):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    best_val_acc = 0.0
    best_state = None
    patience_counter = 0

    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                output = model(X_batch)
                _, predicted = torch.max(output, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()

        val_acc = correct / total if total > 0 else 0.0
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            output = model(X_batch)
            _, predicted = torch.max(output, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return correct / total if total > 0 else 0.0, np.array(all_preds), np.array(all_labels)

## 4. Load All Subjects

In [ ]:
all_subjects = discover_subjects(BIDS_ROOT)
print(f'Found {len(all_subjects)} subjects: {all_subjects}')

subject_data = {}
for subj in all_subjects:
    print(f'Loading sub-{subj} ...', end=' ')
    result = load_subject_data(subj)
    if result is not None:
        subject_data[subj] = result
        print(f'OK ({result["data"].shape[0]} trials, {result["n_channels"]} channels, {result["n_samples"]} samples)')
    else:
        print('FAILED')

print(f'\nSuccessfully loaded {len(subject_data)} subjects')

## 5. Within-Subject Classification

Train EEGNet per subject with 5-fold cross-validation.

In [ ]:
def run_within_subject(subject_id, data_dict, task_name, label_col, n_classes):
    data = data_dict['data']
    meta = data_dict['metadata']
    n_channels = data_dict['n_channels']
    n_samples = data_dict['n_samples']

    le = LabelEncoder()
    labels = le.fit_transform(meta[label_col].values)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_accs = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(data, labels)):
        X_train, X_test = data[train_idx], data[test_idx]
        y_train, y_test = labels[train_idx], labels[test_idx]

        X_train_norm, X_test_norm = normalize_data(X_train, X_test)

        X_train_t = torch.FloatTensor(X_train_norm).unsqueeze(1).to(DEVICE)
        y_train_t = torch.LongTensor(y_train).to(DEVICE)
        X_test_t = torch.FloatTensor(X_test_norm).unsqueeze(1).to(DEVICE)
        y_test_t = torch.LongTensor(y_test).to(DEVICE)

        train_dataset = TensorDataset(X_train_t, y_train_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

        model = EEGNet(n_channels, n_samples, n_classes).to(DEVICE)
        model = train_model(model, train_loader, test_loader)
        acc, _, _ = evaluate_model(model, test_loader)
        fold_accs.append(acc)

    return {
        'Subject': subject_id,
        'Task': task_name,
        'Accuracy': np.mean(fold_accs),
        'Std': np.std(fold_accs),
        'Fold_Accs': fold_accs,
    }

In [ ]:
tasks = {
    'Frequency': {'label_col': 'Frequency', 'n_classes': 3},
    'Transparency': {'label_col': 'Transparency', 'n_classes': 2},
    'Color': {'label_col': 'Color', 'n_classes': 2},
}

within_results = []
for subj, data_dict in subject_data.items():
    for task_name, task_cfg in tasks.items():
        print(f'Within-subject: sub-{subj} / {task_name}', end=' ')
        result = run_within_subject(
            subj, data_dict, task_name,
            task_cfg['label_col'], task_cfg['n_classes'],
        )
        if result is not None:
            within_results.append(result)
            print(f'Acc: {result["Accuracy"]:.4f} +/- {result["Std"]:.4f}')
        else:
            print('SKIPPED')

df_within = pd.DataFrame(within_results)
print('\n=== Within-Subject Summary ===')
print(df_within.groupby('Task')[['Accuracy', 'Std']].agg(['mean', 'std']))

### 5.1 Within-Subject Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, task_name in zip(axes, tasks.keys()):
    task_df = df_within[df_within['Task'] == task_name].sort_values('Subject')
    plt.figure(figsize=(18, 6))
    ax.bar(range(len(task_df)), task_df['Accuracy'], yerr=task_df['Std'],
           capsize=3, color=sns.color_palette('Set2', len(task_df)))
    ax.set_xticks(range(len(task_df)))
    ax.set_xticklabels(task_df['Subject'])
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Within-Subject {task_name} Classification')
    ax.set_ylim(0, 1.05)
    mean_acc = task_df['Accuracy'].mean()
    ax.axhline(mean_acc, color='r', linestyle='--', alpha=0.7, label=f'Mean: {mean_acc:.3f}')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Cross-Subject Classification

5-fold GroupKFold (grouped by subject) ensures no subject appears in both training and test sets.

In [ ]:
def run_cross_subject(all_subject_data, task_name, label_col, n_classes):
    all_X = []
    all_y = []
    all_subjects_groups = []

    for subj, data_dict in all_subject_data.items():
        all_X.append(data_dict['data'])
        le = LabelEncoder()
        all_y.extend(le.fit_transform(data_dict['metadata'][label_col].values))
        all_subjects_groups.extend([subj] * data_dict['data'].shape[0])

    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    groups = np.array(all_subjects_groups)

    n_channels = X.shape[1]
    n_samples = X.shape[2]

    gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        print(f'  Fold {fold + 1}/5: train={len(train_idx)}, test={len(test_idx)}')

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train_norm, X_test_norm = normalize_data(X_train, X_test)

        X_train_t = torch.FloatTensor(X_train_norm).unsqueeze(1).to(DEVICE)
        y_train_t = torch.LongTensor(y_train).to(DEVICE)
        X_test_t = torch.FloatTensor(X_test_norm).unsqueeze(1).to(DEVICE)
        y_test_t = torch.LongTensor(y_test).to(DEVICE)

        train_dataset = TensorDataset(X_train_t, y_train_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

        model = EEGNet(n_channels, n_samples, n_classes).to(DEVICE)
        model = train_model(model, train_loader, test_loader, n_epochs=150, patience=20)
        acc, preds, labels = evaluate_model(model, test_loader)

        test_subjects = groups[test_idx]
        per_subj = {}
        for s in np.unique(test_subjects):
            mask = test_subjects == s
            per_subj[s] = (preds[mask] == labels[mask]).mean()

        fold_results.append({
            'Fold': fold + 1,
            'Accuracy': acc,
            'Test_Subjects': list(np.unique(test_subjects)),
            'Per_Subject_Acc': per_subj,
        })

    all_accs = [r['Accuracy'] for r in fold_results]
    return {
        'Task': task_name,
        'Mean_Accuracy': np.mean(all_accs),
        'Std_Accuracy': np.std(all_accs),
        'Fold_Results': fold_results,
    }

In [ ]:
cross_results = []
for task_name, task_cfg in tasks.items():
    print(f'\nCross-subject: {task_name}')
    result = run_cross_subject(
        subject_data, task_name,
        task_cfg['label_col'], task_cfg['n_classes'],
    )
    cross_results.append(result)
    print(f'  Mean Accuracy: {result["Mean_Accuracy"]:.4f} +/- {result["Std_Accuracy"]:.4f}')

print('\n=== Cross-Subject Summary ===')
for r in cross_results:
    print(f'{r["Task"]}: {r["Mean_Accuracy"]:.4f} +/- {r["Std_Accuracy"]:.4f}')

### 6.1 Cross-Subject Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, result in zip(axes, cross_results):
    fold_accs = [r['Accuracy'] for r in result['Fold_Results']]
    ax.bar(range(len(fold_accs)), fold_accs, color=sns.color_palette('Set2', len(fold_accs)))
    ax.set_xticks(range(len(fold_accs)))
    ax.set_xticklabels([f'Fold {i + 1}' for i in range(len(fold_accs))])
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Cross-Subject {result["Task"]} (Mean: {result["Mean_Accuracy"]:.3f})')
    ax.set_ylim(0, 1.05)
    ax.axhline(result['Mean_Accuracy'], color='r', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 7. Within-Subject vs Cross-Subject Comparison

In [ ]:
comparison_rows = []
for task_name in tasks.keys():
    within_task = df_within[df_within['Task'] == task_name]
    cross_task = next(r for r in cross_results if r['Task'] == task_name)

    comparison_rows.append({
        'Task': task_name,
        'Within_Mean': within_task['Accuracy'].mean(),
        'Within_Std': within_task['Accuracy'].std(),
        'Cross_Mean': cross_task['Mean_Accuracy'],
        'Cross_Std': cross_task['Std_Accuracy'],
    })

df_comparison = pd.DataFrame(comparison_rows)
print('=== Within vs Cross Subject Comparison ===')
print(df_comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_comparison))
width = 0.35
bars1 = ax.bar(x - width / 2, df_comparison['Within_Mean'], width, yerr=df_comparison['Within_Std'],
               label='Within-Subject', capsize=5, color='#2196F3')
bars2 = ax.bar(x + width / 2, df_comparison['Cross_Mean'], width, yerr=df_comparison['Cross_Std'],
               label='Cross-Subject', capsize=5, color='#FF9800')
ax.set_ylabel('Accuracy')
ax.set_title('EEGNet: Within-Subject vs Cross-Subject Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Task'])
ax.set_ylim(0, 1.05)

within_std = df_comparison['Within_Std'].values
cross_std = df_comparison['Cross_Std'].values

for i, bar in enumerate(bars1):
    y_pos = bar.get_height() + within_std[i] + 0.01
    ax.text(bar.get_x() + bar.get_width() / 2., y_pos,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for i, bar in enumerate(bars2):
    y_pos = bar.get_height() + cross_std[i] + 0.01
    ax.text(bar.get_x() + bar.get_width() / 2., y_pos,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()